In [104]:
from folktexts.llm_utils import load_model_tokenizer
from folktexts import TransformersLLMClassifier
from folktexts.acs import ACSDataset

import pandas as pd
from folktexts.task import TaskMetadata
from folktexts.prompting import ACS_TASK_DESCRIPTION
from folktexts.qa_interface import QAInterface
from folktexts.qa_interface import MultipleChoiceQA

from functools import partial
import dataclasses
from dataclasses import dataclass




### Load dataset

In [2]:
acs_task_name = "ACSIncome"     # Name of the benchmark ACS task to use
acs_task = TaskMetadata.get_task(acs_task_name)
dataset = ACSDataset.make_from_task(acs_task_name, cache_dir='./data')   # use `.subsample(0.01)` to get faster approximate results
X_test, y_test = dataset.get_test()
example_row = X_test.iloc[0]

Loading ACS data...


### Load Model

In [54]:
# Load transformers model
model, tokenizer = load_model_tokenizer("gpt2")   # using tiny model as an example

# Create an object that classifies data using an LLM
clf = TransformersLLMClassifier(
    model=model,
    tokenizer=tokenizer,
    task=acs_task_name,
    #encode_row=
)

print(clf._encode_row(example_row))

/Users/mgorecki/opt/miniconda3/envs/llm-py311/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


The following data corresponds to a survey respondent. The survey was conducted among US residents in 2018. Please answer the question based on the information provided. The data provided is enough to reach an approximate answer.

Information:
- age is 23 years old
- class of worker is Working for a for-profit private company or organization
- highest educational attainment is Some college, 1 or more years, no degree
- marital status is Never married
- occupation is Retail salespersons
- place of birth is California
- relationship to the reference person in the survey is Grandchild
- usual number of hours worked per week is 20 hours
- sex is Female
- race is Some other race alone (non-White)

Question: What is this person's estimated yearly income?
A. Below $50,000.
B. Above $50,000.
Answer:


In [55]:
## Default encode row (orginal) (in folktexts/classifier/base.by - LLMClassifier, use as default encode_row_prompt from folktexts.prompting)
def encode_row_prompt(
    row: pd.Series,
    task: TaskMetadata,
    question: QAInterface = None,
    custom_prompt_prefix: str = None,
    add_task_description: bool = True,
) -> str:
    """Encode a question regarding a given row."""
    # Get the question to ask
    question = question or task.question
    return (
        (ACS_TASK_DESCRIPTION + "\n" if add_task_description else "")
        + (f"\n{custom_prompt_prefix}\n" if custom_prompt_prefix else "")
        + f"""\
Information:
{task.get_row_description(row)} ## Custom Change: make get_row_description not a function of the task

{question.get_question_prompt()}""")

Issues with default 'get_row_description()':
- function is task dependent
- function returns string with bullet list of fixed style - not clear how to change 

Changes so far 
- define 'get_row_description()' in folktexts.prompting that takes in style arguments such as connector_verb, bullet list or text

#### Write custom encoding function
Test if overwriting encode_row_prompt does the job?

In [87]:
def serialize_row(
    row: pd.Series,
    task: TaskMetadata,
    style: str = 'bullet',
    connector: str = "is",
    standardized_sentence=True,
):
    if style == "bullet":
        return (
            "\n".join(
                [
                    "- "
                    + f"{task.cols_to_text[col].short_description} {connector} {task.cols_to_text[col].value_map(val)}"
                    for (col, val) in row.items()
                ]
            )
            + "\n"
        )
    elif style == "text":
        return (
            " ".join(
                [
                    (
                        f"The {task.cols_to_text[col].short_description} {connector} {task.cols_to_text[col].value_map(val)}."
                        if standardized_sentence
                        else task.cols_to_text[col]._verbalize(val)
                    )
                    for (col, val) in row.items()
                ]
            )
            + "\n"
        )
    else:
        raise NotImplementedError(
            "Style not implemented, currently only 'bullet' list and 'text' are supported."
        )


def custom_encode_row_prompt(
    row: pd.Series,
    task: TaskMetadata,
    question: QAInterface = None,
    custom_prompt_prefix: str = None,
    add_task_description: bool = True,
    custom_prompt_suffix: str = None,
    **serialize_kwargs
) -> str:
    """Encode a question regarding a given row."""
    if custom_prompt_prefix and custom_prompt_prefix[-1] != "\n":
        custom_prompt_prefix = custom_prompt_prefix + "\n"
    context = (ACS_TASK_DESCRIPTION if add_task_description else "") + (
        f"\n{custom_prompt_prefix}" if custom_prompt_prefix else ""
    )
    # ensure only feature defined for the task are used
    row = row[task.features]
    serialized_row = serialize_row(row, task, **serialize_kwargs)
    question = question or task.question
    return (
        context
        + ("\n" if len(context) > 0 else "")
        + f"Information:\n{serialized_row}"
        + "\n"
        + f"{question.get_question_prompt()}"
        + (custom_prompt_suffix if custom_prompt_suffix else "")
    )

In [94]:
clf = TransformersLLMClassifier(
    model=model,
    tokenizer=tokenizer,
    task=acs_task,#acs_task_name,
    encode_row=partial(custom_encode_row_prompt, task=acs_task, add_task_description=True, style='bullet', connector='is')
)

print(clf._encode_row(X_test.iloc[0]))

The following data corresponds to a survey respondent. The survey was conducted among US residents in 2018. Please answer the question based on the information provided. The data provided is enough to reach an approximate answer.

Information:
- age is 23 years old
- class of worker is Working for a for-profit private company or organization
- highest educational attainment is Some college, 1 or more years, no degree
- marital status is Never married
- occupation is Retail salespersons
- place of birth is California
- relationship to the reference person in the survey is Grandchild
- usual number of hours worked per week is 20 hours
- sex is Female
- race is Some other race alone (non-White)

Question: What is this person's estimated yearly income?
A. Below $50,000.
B. Above $50,000.
Answer:


#### Model-Specific Encodings Encodings?

TABULA introduces extra special tokens used for prompting. Can we adapt the prompt accordingly?

In [83]:
# TABULA
# Predict the value of weather: ||sun||rain||snow|| The date is 2015-03-22. The precipitation is 1.0. The temp_max is 11.699. What is the value of weather? ||sun||rain||snow|| <|endinput|> rain<|endcompletion|>Predict the value of weather: ||sun||rain|| snow|| The date is 2015-09-19. The precipitation is 1.0. The temp_max is 14.722. What is the value of weather? ||sun||rain|| snow||<|endinput|>sun<|endcompletion|>

In [112]:
@dataclass(frozen=True, eq=True)    # NOTE: kw_only=True requires Python 3.10
class TabulaMultipleChoiceQA(MultipleChoiceQA):
     choice_delimiter = '||'

     def get_question_prompt(self) -> str:
        choice_str = self.choice_delimiter+self.choice_delimiter.join([f"{key}. {choice.text}."  for key, choice in self.key_to_choice.items()])+self.choice_delimiter

        return (f"""\
Question: {self.text} {choice_str}
Answer:<|endinput|>""")


tabula_question = TabulaMultipleChoiceQA(
    column=acs_task.target, 
    text = acs_task.question.text,
    choices = acs_task.question.choices,
)

In [113]:
print(tabula_question.get_question_prompt()) # idea: use choice_delimiter (default  \n, tabula: ||)

Question: What is this person's estimated yearly income? ||A. Below $50,000.||B. Above $50,000.||
Answer:<|endinput|>


In [114]:
# choice_delimiter = '||'
encoding_choices =tabula_question.choice_delimiter+tabula_question.choice_delimiter.join([f"{key}. {choice.text}."  for key, choice in acs_task.question.key_to_choice.items()])+tabula_question.choice_delimiter

# tabula_question = MultipleChoiceQA(
#     column=acs_task.target, 
#     text = f"{acs_task.question.text} {encoding_choices}",
#     choices = acs_task.question.choices,
# )
# ## requires to overwrite how question is translated into prompt

print(custom_encode_row_prompt(
    example_row,
    acs_task,
    custom_prompt_prefix=f"Predict the value of {acs_task.cols_to_text[acs_task.target].short_description}: {encoding_choices}", ## Choices = natural text or A/B or numeric values?
    question=tabula_question, 
    custom_prompt_suffix='<|endinput|>'
))

The following data corresponds to a survey respondent. The survey was conducted among US residents in 2018. Please answer the question based on the information provided. The data provided is enough to reach an approximate answer.

Predict the value of yearly income: ||A. Below $50,000.||B. Above $50,000.||

Information:
- age is 23 years old
- class of worker is Working for a for-profit private company or organization
- highest educational attainment is Some college, 1 or more years, no degree
- marital status is Never married
- occupation is Retail salespersons
- place of birth is California
- relationship to the reference person in the survey is Grandchild
- usual number of hours worked per week is 20 hours
- sex is Female
- race is Some other race alone (non-White)

Question: What is this person's estimated yearly income? ||A. Below $50,000.||B. Above $50,000.||
Answer:<|endinput|><|endinput|>


In [103]:
print(tabula_question.get_question_prompt())

                Question: What is this person's estimated yearly income? ||A. Below $50,000.||B. Above $50,000.|| ||A. Below $50,000.||B. Above $50,000.||
                Answer:<|endinput|>
